In [2]:
# libraries

import requests
from bs4 import BeautifulSoup
import time
import re
import csv
from datetime import datetime

In [18]:

class BamaSamandScraper:
    def __init__(self):
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
            'Accept-Language': 'fa-IR,fa;q=0.9,en;q=0.8',
        }
        self.cars_data = []
        self.seen_urls = set()
        
    def convert_persian_numbers(self, text):
        """Convert Persian numbers to English"""
        if not text:
            return text
        
        persian_digits = '۰۱۲۳۴۵۶۷۸۹'
        english_digits = '0123456789'
        
        for pr, en in zip(persian_digits, english_digits):
            text = text.replace(pr, en)
        return text
    
    def extract_year_from_card(self, card_text, car_url=""):
        """Extract year from card text with multiple patterns"""
        # Method 1: Try to extract year from URL 
        if car_url:
            
            url_year_match = re.search(r'-(\d{4})$', car_url)
            if url_year_match:
                year = url_year_match.group(1)
                # Convert if Persian digits
                year = self.convert_persian_numbers(year)
                year_int = int(year)
                if 1380 <= year_int <= 1405:
                    return year
        
        # Method 2: Try Persian years directly
        persian_year_match = re.search(r'[۱۳۱۴][۰-۹]{3}', card_text)
        if persian_year_match:
            year = self.convert_persian_numbers(persian_year_match.group())
            return year
        
        # Method 3: Try regular digits
        year_match = re.search(r'\b(13[0-9]{2}|14[0-9]{2})\b', card_text)
        if year_match:
            return year_match.group(1)
        
        # Method 4: Look for any 4-digit number in range
        all_numbers = re.findall(r'\d{4}', card_text)
        for num in all_numbers:
            try:
                num_int = int(num)
                if 1380 <= num_int <= 1405:
                    return num
            except:
                continue
        
        # Method 5: Try to find year in Persian text format 
        model_match = re.search(r'مدل\s*([۱۳۱۴][۰-۹]{3})', card_text)
        if model_match:
            return self.convert_persian_numbers(model_match.group(1))
        
        # Method 6: Look for year with "سال" 
        year_match2 = re.search(r'سال\s*[۱۳۱۴][۰-۹]{3}', card_text)
        if year_match2:
            year_text = re.search(r'[۱۳۱۴][۰-۹]{3}', year_match2.group())
            if year_text:
                return self.convert_persian_numbers(year_text.group())
        
        return None
    
    def extract_price_from_listing(self, card_text):
        """Extract price from listing card"""
        if 'قیمت توافقی' in card_text:
            return "Negotiable (قیمت توافقی)"
        
        price_patterns = [
            r'(\d{1,3}(?:,\d{3})*)\s*تومان',
            r'(\d+(?:,\d+)*)\s*تومان',
        ]
        
        for pattern in price_patterns:
            match = re.search(pattern, card_text)
            if match:
                price = match.group(1).replace(',', '')
                try:
                    return f"{int(price):,} تومان"
                except:
                    return f"{price} تومان"
        return "N/A"
    
    def extract_mileage_from_listing(self, card_text):
        """Extract mileage from listing card"""
        if 'صفر کیلومتر' in card_text:
            return "0 km (صفر کیلومتر)"
        
        mileage_patterns = [
            r'(\d{1,3}(?:,\d{3})*)\s*kilometers?',
            r'(\d{1,3}(?:,\d{3})*)\s*km',
            r'(\d{1,3}(?:,\d{3})*)\s*کیلومتر',
        ]
        
        for pattern in mileage_patterns:
            match = re.search(pattern, card_text, re.IGNORECASE)
            if match:
                mileage = match.group(1).replace(',', '')
                try:
                    return f"{int(mileage):,} km"
                except:
                    return f"{mileage} km"
        return "N/A"
    
    def extract_color_and_transmission_from_detail(self, soup):
        """Extract both color and transmission from detail page"""
        specs = {
            'color': 'N/A',
            'transmission': 'N/A'
        }
        
        # Find all spans with the specific class
        spec_spans = soup.find_all('span', class_='inline-block text-center w-full text-sm leading-5 font-semibold')
        
        for span in spec_spans:
            spec_text = span.get_text(strip=True)
            
            # Check for color
            color_keywords = ['سفید', 'مشکی', 'نقره‌ای', 'طوسی', 'آبی', 'قرمز', 'سبز', 'زرد', 
                             'کرم', 'قهوه‌ای', 'نارنجی', 'بنفش', 'سرمه‌ای', 'طلایی', 'فیروزه‌ای']
            if spec_text in color_keywords:
                specs['color'] = spec_text
            
            # Check for transmission
            if spec_text == 'اتوماتیک':
                specs['transmission'] = 'Automatic (اتوماتیک)'
            elif 'دنده' in spec_text:
                specs['transmission'] = 'Manual (دنده‌ای)'
        
        return specs
    
    def extract_description_from_detail(self, soup):
        """Extract description from detail page"""
        # Find the div that contains both the heading and description
        desc_div = soup.find('div', class_='flex flex-col gap-2 items-start self-stretch')
        
        if desc_div:
            all_spans = desc_div.find_all('span')
            
            if len(all_spans) >= 2:
                heading_span = all_spans[0]
                heading_text = heading_span.get_text(strip=True)
                
                if heading_text == 'توضیحات':
                    description_span = all_spans[1]
                    description_text = description_span.get_text(strip=True)
                    
                    if description_text:
                        description_text = re.sub(r'\s+', ' ', description_text)
                        return description_text
        
        return "N/A"
    
    def extract_model_from_listing(self, card):
        """Extract car model from the specific span in listing card"""
        model_span = card.find('span', class_='inline-block text-right w-full text-base leading-6 font-semibold truncate text-neutral-10 mb-0.5')
        
        if model_span:
            model_text = model_span.get_text(strip=True)
            return model_text
        
        fallback_span = card.find('span', class_='font-semibold')
        if fallback_span:
            return fallback_span.get_text(strip=True)
        
        return "N/A"
    
    def extract_submodel_from_listing(self, card):
        """Extract car submodel/trim from the listing card"""
        subtitle_span = card.find('span', class_='inline-block text-right w-full text-xs leading-4 font-normal truncate text-neutral-10 mb-1.5')
        
        if subtitle_span:
            return subtitle_span.get_text(strip=True)
        
        return "N/A"
    
    def get_car_details(self, car_url):
        """Visit car detail page to extract color, transmission, and description"""
        try:
            full_url = f"https://bama.ir{car_url}" if not car_url.startswith('http') else car_url
            response = requests.get(full_url, headers=self.headers, timeout=15)
            
            if response.status_code != 200:
                return {
                    'color': 'N/A', 
                    'transmission': 'N/A', 
                    'detailed_description': 'N/A'
                }, False
            
            soup = BeautifulSoup(response.text, 'html.parser')
            specs = self.extract_color_and_transmission_from_detail(soup)
            detailed_desc = self.extract_description_from_detail(soup)
            
            return {
                'color': specs['color'],
                'transmission': specs['transmission'],
                'detailed_description': detailed_desc
            }, True
            
        except Exception as e:
            print(f"       Error: {e}")
            return {
                'color': 'N/A', 
                'transmission': 'N/A', 
                'detailed_description': 'N/A'
            }, False
    
    def scrape_listings_and_details(self, target_count=50):
        """Scrape listings and details in one efficient pass"""
        base_url = "https://bama.ir/car/samand"
        page = 1
        max_pages = 30
        consecutive_empty_pages = 0
        
        print(f"\n{'='*80}")
        print(" SCRAPING SAMAND CARS (1386+)")
        print(f"{'='*80}")
        print(f" Target: {target_count} cars")
        print(f" Started: {datetime.now().strftime('%H:%M:%S')}\n")
        
        while len(self.cars_data) < target_count and page <= max_pages:
            url = f"{base_url}?page={page}" if page > 1 else base_url
            
            try:
                print(f" Page {page} - Fetching listings from: {url}")
                response = requests.get(url, headers=self.headers, timeout=15)
                
                if response.status_code != 200:
                    print(f"    Failed with status: {response.status_code}")
                    break
                
                soup = BeautifulSoup(response.text, 'html.parser')
                
                all_links = soup.find_all('a', href=True)
                car_cards = [link for link in all_links if link['href'].startswith('/car/detail-')]
                
                print(f"    Found {len(car_cards)} total car cards on page {page}")
                
                if not car_cards:
                    print(f"    No car cards found - stopping")
                    break
                
                cars_added_this_page = 0
                
                for idx, card in enumerate(car_cards):
                    if len(self.cars_data) >= target_count:
                        break
                    
                    car_url = card['href']
                    if car_url in self.seen_urls:
                        continue
                    self.seen_urls.add(car_url)
                    
                    card_text = card.get_text()
                    
                    # Extract year with URL as additional source
                    year = self.extract_year_from_card(card_text, car_url)
                    
                    if not year:
                        
                        if page >= 2 and idx < 5:
                            print(f"    DEBUG - No year found for card {idx+1} on page {page}")
                            print(f"      URL: {car_url}")
                            print(f"      Text preview: {card_text[:200]}...")
                        continue
                    
                    year_int = int(year)
                    if year_int < 1386:
                        if page >= 2 and idx < 5:
                            print(f"    DEBUG - Year {year} < 1386, skipping")
                        continue
                    
                    price = self.extract_price_from_listing(card_text)
                    mileage = self.extract_mileage_from_listing(card_text)
                    model = self.extract_model_from_listing(card)
                    submodel = self.extract_submodel_from_listing(card)
                    
                    full_model = f"{model} - {submodel}" if submodel != "N/A" else model
                    
                    print(f"\n Car #{len(self.cars_data) + 1}:")
                    print(f"   Model: {full_model}")
                    print(f"   Year: {year} (from page {page})")
                    print(f"   Price: {price[:40]}")
                    print(f"   Mileage: {mileage}")
                    
                    details, success = self.get_car_details(car_url)
                    
                    description_parts = [full_model]
                    if details['detailed_description'] != 'N/A':
                        description_parts.append(f"توضیحات: {details['detailed_description']}")
                    
                    full_description = ' | '.join(description_parts)
                    
                    car_data = {
                        'Model': full_model,
                        'Production year': year,
                        'Price': price,
                        'Mileage': mileage,
                        'Color': details['color'],
                        'Transmission type': details['transmission'],
                        'Description': full_description,
                        'URL': f"https://bama.ir{car_url}",
                        'Scraped at': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                    }
                    
                    self.cars_data.append(car_data)
                    cars_added_this_page += 1
                    
                    print(f"   Color: {details['color']}")
                    print(f"   Transmission: {details['transmission']}")
                    if details['detailed_description'] != 'N/A':
                        desc_preview = details['detailed_description'][:80] if len(details['detailed_description']) > 80 else details['detailed_description']
                        print(f"   Description: {desc_preview}...")
                    else:
                        print(f"   Description: None provided")
                    print(f"   Progress: {len(self.cars_data)}/{target_count}")
                    
                    time.sleep(1)
                
                print(f"\n    Page {page} summary: Added {cars_added_this_page} cars (Total: {len(self.cars_data)}/{target_count})")
                
                if cars_added_this_page == 0:
                    consecutive_empty_pages += 1
                    print(f"    No cars added on page {page} (consecutive empty pages: {consecutive_empty_pages})")
                    
                    if consecutive_empty_pages >= 3:
                        print(f"    Three consecutive empty pages - stopping")
                        break
                else:
                    consecutive_empty_pages = 0
                
                page += 1
                time.sleep(2)
                
            except Exception as e:
                print(f"    Error on page {page}: {e}")
                break
        
        return self.cars_data
    
    def save_to_csv(self, filename='samand_cars_complete.csv'):
        """Save complete data to CSV"""
        if not self.cars_data:
            print("No data to save")
            return
        
        fieldnames = ['Model', 'Production year', 'Price', 'Mileage', 'Color', 
                     'Transmission type', 'Description', 'URL', 'Scraped at']
        
        with open(filename, 'w', newline='', encoding='utf-8-sig') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(self.cars_data)
        
        print(f"\n Data saved to '{filename}'")
    
    def display_statistics(self):
        """Display statistics about scraped data"""
        if not self.cars_data:
            return
        
        print(f"\n{'='*80}")
        print(" SCRAPING STATISTICS")
        print(f"{'='*80}")
        
        models = {}
        for car in self.cars_data:
            model = car['Model']
            if model != 'N/A':
                base_model = model.split(' - ')[0] if ' - ' in model else model
                models[base_model] = models.get(base_model, 0) + 1
        
        if models:
            print(f"\n Model Distribution:")
            for model, count in sorted(models.items(), key=lambda x: x[1], reverse=True):
                print(f"   {model}: {count} cars ({count/len(self.cars_data)*100:.1f}%)")
        
        colors = {}
        for car in self.cars_data:
            color = car['Color']
            if color != 'N/A':
                colors[color] = colors.get(color, 0) + 1
        
        if colors:
            print(f"\n Color Distribution:")
            for color, count in sorted(colors.items(), key=lambda x: x[1], reverse=True):
                print(f"   {color}: {count} cars ({count/len(self.cars_data)*100:.1f}%)")
        
        transmissions = {}
        for car in self.cars_data:
            trans = car['Transmission type']
            if trans != 'N/A':
                transmissions[trans] = transmissions.get(trans, 0) + 1
        
        if transmissions:
            print(f"\n Transmission Type Distribution:")
            for trans, count in transmissions.items():
                print(f"   {trans}: {count} cars ({count/len(self.cars_data)*100:.1f}%)")
        
        colors_found = sum(1 for c in self.cars_data if c['Color'] != 'N/A')
        trans_found = sum(1 for c in self.cars_data if c['Transmission type'] != 'N/A')
        
        print(f"\n Extraction Success Rates:")
        print(f"   Color: {colors_found}/{len(self.cars_data)} ({colors_found/len(self.cars_data)*100:.1f}%)")
        print(f"   Transmission: {trans_found}/{len(self.cars_data)} ({trans_found/len(self.cars_data)*100:.1f}%)")
        
        years = {}
        for car in self.cars_data:
            year = car['Production year']
            years[year] = years.get(year, 0) + 1
        
        print(f"\n Production Year Distribution (Top 10):")
        for year in sorted(years.keys(), reverse=True)[:10]:
            print(f"   {year}: {years[year]} cars")
        
        print(f"{'='*80}")
    
    def display_sample_data(self):
        """Display sample data to verify quality"""
        print(f"\n{'='*80}")
        print(" SAMPLE DATA (First 3 cars)")
        print(f"{'='*80}")
        
        for i, car in enumerate(self.cars_data[:3], 1):
            print(f"\n Car #{i}:")
            print(f"   Model: {car['Model']}")
            print(f"   Year: {car['Production year']}")
            print(f"   Price: {car['Price']}")
            print(f"   Color: {car['Color']}")
            print(f"   Transmission: {car['Transmission type']}")
            print(f"   Description:")
            desc = car['Description']
            if len(desc) > 400:
                print(f"      {desc[:400]}...")
            else:
                print(f"      {desc}")
    
    def run(self, target_count=50):
        """Main method to run the complete scraper"""
        self.scrape_listings_and_details(target_count)
        
        print(f"\n{'='*80}")
        print(" SCRAPING COMPLETED!")
        print(f"{'='*80}")
        print(f" Finished at: {datetime.now().strftime('%H:%M:%S')}")
        print(f" Total cars scraped: {len(self.cars_data)}")
        
        if self.cars_data:
            #self.display_statistics()
            #self.display_sample_data()
            self.save_to_csv()
        else:
            print("\n No data was scraped. Please check the website structure or your internet connection.")

if __name__ == "__main__":
    scraper = BamaSamandScraper()
    scraper.run(target_count=50)


 SCRAPING SAMAND CARS (1386+)
 Target: 50 cars
 Started: 01:18:27

 Page 1 - Fetching listings from: https://bama.ir/car/samand
    Found 31 total car cards on page 1

 Car #1:
   Model: سمند، سورن
   Year: 1404 (from page 1)
   Price: 2,000,000,000 تومان
   Mileage: 0 km (صفر کیلومتر)
   Color: سفید
   Transmission: Manual (دنده‌ای)
   Description: None provided
   Progress: 1/50

 Car #2:
   Model: سمند، سورن
   Year: 1403 (from page 1)
   Price: 1,750,000,000 تومان
   Mileage: 25,000 km
   Color: مشکی
   Transmission: Manual (دنده‌ای)
   Description: None provided
   Progress: 2/50

 Car #3:
   Model: سمند، سورن
   Year: 1403 (from page 1)
   Price: 1,450,000,000 تومان
   Mileage: 43,000 km
   Color: مشکی
   Transmission: Manual (دنده‌ای)
   Description: با دورود فراوان سورن پلاس موتور ef7 مدل 403 کارکرد 43000 تمام فابریک دو تا کف دس...
   Progress: 3/50

 Car #4:
   Model: سمند، LX
   Year: 1386 (from page 1)
   Price: 600,000,000 تومان
   Mileage: 400,000 km
   Color: کرم
   Tran

>After scraping page 1, the website is blocking me after too many rapid requests, so I couldn't scrap other pages.